# Modeling, Segmentation & Evaluation

## Overview
This notebook focuses on building, comparing, and evaluating machine learning models for the Bank Marketing dataset. Building on the preprocessing and feature engineering steps from previous notebooks, we now develop predictive models to estimate the likelihood of a client subscribing to a term deposit.

In addition to a traditional global modeling approach, this notebook explores a segmentation-based strategy using clusters derived earlier. By comparing a single global model against cluster-specific models, we aim to understand whether segment-level specialization leads to improved predictive performance and better business outcomes.



## Objectives
- Compare multiple candidate model architectures using the training and validation sets  
- Select a primary modeling approach based on validation performance  
- Train a global model using all available data and evaluate performance across clusters  
- Train segment-specific models for each cluster and compare against the global model  
- Perform hyperparameter tuning for both global and segment-specific models using validation data  
- Train final models using selected hyperparameters  
- Apply probability calibration to improve predicted probabilities  
- Conduct cost-sensitive analysis using calibrated probabilities and different decision thresholds  
- Evaluate final models on a held-out test set, including overall and cluster-level performance  



## Dataset Description
The modeling process uses the preprocessed dataset generated in the previous notebook, which includes engineered features such as categorical encodings, binary indicators, and interaction terms. Additionally, each observation is assigned to a cluster derived from unsupervised learning, enabling both global and segment-specific modeling approaches.

The data is split into three subsets:

- **Training set**: used to fit models  
- **Validation set**: used for model selection, hyperparameter tuning, and calibration  
- **Test set**: held out for final unbiased evaluation  

The target variable represents whether a client subscribed to a term deposit.



## Key Considerations
- **Model comparison**: Different model architectures are evaluated to identify the most suitable approach for the problem  
- **Segmentation strategy**: Cluster-based segmentation allows us to investigate heterogeneity in client behavior and model performance  
- **Fairness and consistency**: Performance is evaluated across clusters to detect potential disparities  
- **Overfitting prevention**: Validation data is strictly used for model selection and hyperparameter tuning  
- **Probability calibration**: Raw model probabilities may be poorly calibrated, which can negatively impact decision-making  
- **Business alignment**: Model evaluation incorporates cost-sensitive analysis, considering trade-offs between false positives (e.g., unnecessary calls) and false negatives (missed opportunities)  
- **Final evaluation integrity**: The test set remains untouched until the final evaluation step  



## Outcome
By the end of this notebook, we will have:
- Selected and trained a primary global model  
- Developed segment-specific models tailored to different client clusters  
- Tuned and calibrated all models for improved predictive reliability  
- Compared global and segmented approaches from both predictive and business perspectives  
- Identified optimal decision thresholds based on expected cost  
- Produced a final evaluation on the test set, including overall performance, cluster-level insights, and cost-based metrics  

In [1]:
import pandas as pd
import lightgbm as lgb
import optuna
import joblib

from pathlib import Path
from sklearn.model_selection import ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

from utils.utils import load_dataset, evaluate_model, evaluate_from_probs

RANDOM_STATE = 705

/Users/tonantzinrealrojas/miniforge3/envs/nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# quarto preview 03_training.ipynb --to pdf
# quarto render 03_training.ipynb
# black 03_training.ipynb

## 1. Global architecture selection

In this section, we evaluate a set of candidate model architectures to determine which will be used in the subsequent comparison between a global model (trained on all observations) and segment-specific models (trained per cluster).

We begin by training and assessing five baseline architectures using their default configurations:

1. Logistic Regression  
2. Random Forest  
3. XGBoost  
4. LightGBM  
5. CatBoost  

The goal is to identify a strong primary model that balances predictive performance and generalization before proceeding to more advanced analysis.

In [2]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# from sklearn.metrics import (
#     average_precision_score,
#     f1_score,
#     precision_score,
#     recall_score,
# )

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

Because the outcome variable is highly imbalanced, we apply class weighting across all models to penalize misclassification of subscribers more heavily than non‑subscribers. For tree‑based models, this is implemented via the ratio of negative to positive samples, while linear and ensemble baselines use built‑in balanced class weights. This approach preserves the original data distribution and produces probability estimates suitable for downstream calibration and cost‑based decision analysis.

In [3]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [4]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [5]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

### i. Logistic Regression

In [6]:
log_reg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

log_reg.fit(X_train, y_train)
log_reg_metrics = evaluate_model(log_reg, X_validation, y_validation)

### ii. Random Forest

In [7]:
rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

rf.fit(X_train, y_train)
rf_metrics = evaluate_model(rf, X_validation, y_validation)

### iii. XGBoost

In [8]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)

xgb.fit(X_train, y_train)
xgb_metrics = evaluate_model(xgb, X_validation, y_validation)

### iv. LightGBM

In [9]:
lgbm = lgb.LGBMClassifier(
    objective="binary", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE
)

lgbm.fit(X_train, y_train)
lgbm_metrics = evaluate_model(lgbm, X_validation, y_validation)

[LightGBM] [Info] Number of positive: 3385, number of negative: 25549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001658 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 176
[LightGBM] [Info] Number of data points in the train set: 28934, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116990 -> initscore=-2.021244
[LightGBM] [Info] Start training from score -2.021244


### v. CatBoost

In [10]:
cat = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    class_weights=[1, scale_pos_weight],
    verbose=0,
    random_state=RANDOM_STATE,
)

cat.fit(X_train, y_train)
cat_metrics = evaluate_model(cat, X_validation, y_validation)

### Results comparison

In [13]:
results = pd.DataFrame.from_dict(
    {
        "Logistic Regression": log_reg_metrics,
        "Random Forest": rf_metrics,
        "XGBoost": xgb_metrics,
        "LightGBM": lgbm_metrics,
        "CatBoost": cat_metrics,
    },
    orient="index",
)

results.sort_values(by="AUC_PR", ascending=False)

,AUC_PR,F1,Precision,Recall
LightGBM,0.445832,0.459201,0.362826,0.625296
CatBoost,0.442484,0.459831,0.368159,0.612293
Logistic Regression,0.411748,0.395126,0.287325,0.632388
XGBoost,0.407729,0.434281,0.347795,0.578014
Random Forest,0.313136,0.275808,0.413712,0.206856


On the validation set, LightGBM and CatBoost exhibit very similar performance, with nearly identical F1 scores and only a marginal difference in AUC‑PR (0.446 for LightGBM versus 0.442 for CatBoost). Both models clearly outperform the other architectures and appear well suited to handling the strong class imbalance present in the data.

LightGBM was selected as the primary model architecture because it achieves the highest AUC‑PR overall, indicating slightly better ranking of likely subscribers, while maintaining strong recall and competitive precision. Given the minimal performance gap, this choice is pragmatic rather than definitive; CatBoost remains a strong alternative and could be expected to perform similarly in downstream analysis. **LightGBM** is therefore used for the remainder of the project to provide a consistent baseline for both global and segment‑specific modeling.

## 2. Global model

Once we decided to work with LightGBM, we'll train the global model which will the observations from all clusters.

In [2]:
X_train_, y_train = load_dataset("02", "train")
X_validation_, y_validation = load_dataset("02", "validation")
# X_test_, y_test_ = load_dataset(nb="02", split="test")


train
X shape: (28934, 60)
y shape: (28934,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0



validation
X shape: (7234, 60)
y shape: (7234,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0


In [3]:
cluster_feat = "cluster"

X_train = X_train_.copy()
X_train = X_train.drop(columns=cluster_feat)
print("\nTrain set: ", X_train.shape)
display(X_train.head(3))

X_validation = X_validation_.copy()
X_validation = X_validation.drop(columns=cluster_feat).copy()
print("\nValidation set: ", X_validation.shape)
display(X_validation.head(3))

# X_test = X_test_.copy()
# X_test = X_test.drop(columns=cluster_feat).copy()
# print("\nTest set: ", X_test.shape)
# display(X_test.head(3))


Train set:  (28934, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0



Validation set:  (7234, 59)


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__default,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [4]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"Scale pos weight: {scale_pos_weight:.2f}")

Scale pos weight: 7.55


For hyperparameter tuning, we'll use a Bayesian approach

In [5]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "aucpr",
        "random_state": RANDOM_STATE,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        # hyperparameters to tune
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    y_val_prob = model.predict_proba(X_validation)[:, 1]
    aucpr = average_precision_score(y_validation, y_val_prob)

    return aucpr

In [6]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC-PR:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params

[I 2026-04-10 23:04:27,987] A new study created in memory with name: no-name-e246ef53-0664-4780-888f-c749538ed4bc
Best trial: 0. Best value: 0.44695:   3%|▎         | 1/30 [00:01<00:30,  1.04s/it]

[I 2026-04-10 23:04:29,046] Trial 0 finished with value: 0.44695013721562915 and parameters: {'num_leaves': 84, 'max_depth': 5, 'learning_rate': 0.07431837596167686, 'n_estimators': 331, 'min_child_samples': 159}. Best is trial 0 with value: 0.44695013721562915.


Best trial: 0. Best value: 0.44695:   7%|▋         | 2/30 [00:04<01:08,  2.46s/it]

[I 2026-04-10 23:04:32,490] Trial 1 finished with value: 0.4388096066105165 and parameters: {'num_leaves': 83, 'max_depth': 0, 'learning_rate': 0.02068655937226802, 'n_estimators': 234, 'min_child_samples': 163}. Best is trial 0 with value: 0.44695013721562915.


Best trial: 2. Best value: 0.44942:  10%|█         | 3/30 [00:05<00:50,  1.85s/it]

...

[I 2026-04-10 23:05:15,870] Trial 26 finished with value: 0.4478586712199646 and parameters: {'num_leaves': 33, 'max_depth': 3, 'learning_rate': 0.1901998437319316, 'n_estimators': 477, 'min_child_samples': 21}. Best is trial 16 with value: 0.45746041575095986.
[I 2026-04-10 23:05:16,035] Trial 27 finished with value: 0.41034145926368126 and parameters: {'num_leaves': 74, 'max_depth': 1, 'learning_rate': 0.12777240113068722, 'n_estimators': 249, 'min_child_samples': 92}. Best is trial 16 with value: 0.45746041575095986.


Best trial: 16. Best value: 0.45746:  97%|█████████▋| 29/30 [00:49<00:00,  1.04it/s]

[I 2026-04-10 23:05:17,761] Trial 28 finished with value: 0.44201593069925194 and parameters: {'num_leaves': 44, 'max_depth': 5, 'learning_rate': 0.09216093320614571, 'n_estimators': 557, 'min_child_samples': 67}. Best is trial 16 with value: 0.45746041575095986.


Best trial: 16. Best value: 0.45746: 100%|██████████| 30/30 [00:50<00:00,  1.68s/it]

[I 2026-04-10 23:05:18,429] Trial 29 finished with value: 0.44781197210029083 and parameters: {'num_leaves': 23, 'max_depth': 4, 'learning_rate': 0.1618314366611486, 'n_estimators': 327, 'min_child_samples': 84}. Best is trial 16 with value: 0.45746041575095986.
Best AUC-PR: 0.45746041575095986
Best parameters: {'num_leaves': 31, 'max_depth': 4, 'learning_rate': 0.09908788645704905, 'n_estimators': 348, 'min_child_samples': 55}


We train the final global base model with the best parameters

In [7]:
global_model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    **best_params
)

global_model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.09908788645704905
,n_estimators,348
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,55


Now we do the probabilities calibration

In [8]:
# raw probabilities on validation
val_probs_raw = global_model.predict_proba(X_validation)[:, 1]

# isotonic regression calibration
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs_raw, y_validation)

# calibrated probabilities
val_probs_calibrated = calibrator.transform(val_probs_raw)

In [9]:
global_validation_metrics = evaluate_from_probs(y_validation, val_probs_calibrated)

print("Global model (validation, calibrated):")
print(global_validation_metrics)

Global model (validation, calibrated):
{'AUC_PR': 0.4457539936060022, 'F1': 0.2846441947565543, 'Precision': 0.6846846846846847, 'Recall': 0.17966903073286053}


In [10]:
cluster_results = {}

for c in sorted(X_validation_[cluster_feat].unique()):
    idx = X_validation_[cluster_feat] == c

    cluster_results[f"cluster_{c}"] = evaluate_from_probs(
        y_validation.loc[idx], val_probs_calibrated[idx]
    )

cluster_results_df = pd.DataFrame(cluster_results).T
cluster_results_df

,AUC_PR,F1,Precision,Recall
cluster_0,0.563676,0.387597,0.757576,0.260417
cluster_1,0.295621,0.171717,0.629630,0.099415
cluster_2,0.380164,0.243478,0.608696,0.152174
cluster_3,0.495403,0.298429,0.686747,0.190635


In [11]:
model_dir = Path("models")

model_name = "global"

# save base model
joblib.dump(global_model, model_dir / f"{model_name}_model.joblib")

# save calibrator
joblib.dump(calibrator, model_dir / f"{model_name}_calibrator.joblib")

# save metadata
metadata = {
    "best_params": best_params,
    "scale_pos_weight": scale_pos_weight,
    "features": list(X_train.columns),
    "random_state": RANDOM_STATE,
}

joblib.dump(metadata, model_dir / f"{model_name}_metadata.joblib")

['models/global_metadata.joblib']

## 3. Cluster 0 model

In [ ]:
cluster_id = 0
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 4. Cluster 1 model

In [ ]:
cluster_id = 1
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 5. Cluster 2 model

In [ ]:
cluster_id = 2
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)

## 6. Cluster 3 model

In [ ]:
cluster_id = 3
X_train, y_train = load_dataset(nb="02", split="train", cluster_id=cluster_id)
X_validation, y_validation = load_dataset(
    nb="02", split="validation", cluster_id=cluster_id
)
X_test, y_test = load_dataset(nb="02", split="test", cluster_id=cluster_id)